# Promoter prediction with the Genomic Intelligence model package

This notebook deploys the Genomic Intelligence promoter model from AWS Marketplace in your own account, scores the sample sequences on a real-time endpoint, runs the same samples as a batch transform job, and deletes everything it created.

**Before you run it**

1. Subscribe to the product on AWS Marketplace (search for *Genomic Intelligence Promoter Prediction*), then choose **Continue to configuration**. The notebook finds the model package for your Region itself.
2. Run it from SageMaker Studio or a notebook instance whose execution role can create SageMaker models, endpoints and transform jobs, and read and write the Region's default SageMaker bucket.
3. You need quota for one `ml.g5.xlarge` endpoint and one `ml.g5.xlarge` transform job in your Region.

**Cost.** The endpoint bills from the moment it is `InService` until the cleanup cell deletes it, at the SageMaker instance rate plus the product's software price. Run the cleanup cell even if something fails.

Research use only. Not for diagnostic or clinical use.

In [ ]:
import json, time
from pathlib import Path

import boto3

session = boto3.session.Session()
region = session.region_name
sm = session.client("sagemaker")
runtime = session.client("sagemaker-runtime")
s3 = session.client("s3")
account = session.client("sts").get_caller_identity()["Account"]

try:
    import sagemaker
    role = sagemaker.get_execution_role()
except Exception:
    raise SystemExit("Run this from SageMaker Studio or a notebook instance, or set `role` to an execution role ARN.")

bucket = f"sagemaker-{region}-{account}"
prefix = "gi-promoter-example"
INSTANCE = "ml.g5.xlarge"
print(region, role, bucket)

## The model package for your Region

AWS Marketplace publishes one model package ARN per Region. The table below is filled in when the listing is published.

In [ ]:
# Filled in on publication: {region: model package ARN}.
MODEL_PACKAGE_ARNS = {
}

if region not in MODEL_PACKAGE_ARNS:
    raise SystemExit(
        f"No model package ARN for {region} yet. Supported: {sorted(MODEL_PACKAGE_ARNS) or 'none listed'}. "
        "Copy the ARN shown under 'Continue to configuration' on the listing into MODEL_PACKAGE_ARNS."
    )
model_package_arn = MODEL_PACKAGE_ARNS[region]
print(model_package_arn)

## Sample data

Three 2,000 bp GRCh38 intervals: the human ACTB and GAPDH promoters, and an intergenic negative control. Each file is a complete request body: `sequence` (A, C, G, T, N; 300 to 500,000 bp) and an optional `sequence_name`. Any other field is rejected with HTTP 400.

In [ ]:
DATA = Path("data")
samples = {p.stem: json.loads(p.read_text()) for p in sorted((DATA / "input").glob("*.json"))}
for name, body in samples.items():
    print(f"{name}: {len(body['sequence'])} bp")

## Deploy a real-time endpoint

The model runs with network isolation, as every Marketplace model package does. This takes about 10 minutes.

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
model_name = f"gi-promoter-{stamp}"
endpoint_name = f"gi-promoter-{stamp}"

sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={"ModelPackageName": model_package_arn},
    EnableNetworkIsolation=True,
)
sm.create_endpoint_config(
    EndpointConfigName=endpoint_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InstanceType": INSTANCE,
        "InitialInstanceCount": 1,
    }],
)
sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_name)
sm.get_waiter("endpoint_in_service").wait(EndpointName=endpoint_name)
print("InService:", endpoint_name)

## Score the samples

The response gives every window's probability (`window_details`), the windows at 0.5 or more (`regions`), and ready-made BED and bedGraph text (`formats`).

In [ ]:
results = {}
for name, body in samples.items():
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(body),
    )
    results[name] = json.loads(response["Body"].read())
    probs = [round(w["probability"], 4) for w in results[name]["window_details"]]
    print(f"{name}: windows {probs}, promoter windows {results[name]['summary']['promoter_windows']}")

Path(f"{'ACTB_promoter'}.bed").write_text(results["ACTB_promoter"]["formats"]["bed"])

There is no threshold parameter. To use your own cut-off, filter the windows yourself:

In [ ]:
cutoff = 0.9
for name, r in results.items():
    hits = [(w["prediction_start"], w["prediction_end"]) for w in r["window_details"] if w["probability"] >= cutoff]
    print(name, hits)

Compare with the published responses. Everything should match except the last few bits of each probability, which depend on the GPU.

In [ ]:
for name, r in results.items():
    expected = json.loads((DATA / "output" / f"{name}.json").read_text())
    delta = max(abs(a["probability"] - b["probability"]) for a, b in zip(r["window_details"], expected["window_details"]))
    print(f"{name}: max probability difference {delta:.1e}")

## Batch transform

Each S3 object is one request body, passed whole to the model (`SplitType: None`), and each response comes back as its own `.out` object.

In [ ]:
for name, body in samples.items():
    s3.put_object(Bucket=bucket, Key=f"{prefix}/input/{name}.json", Body=json.dumps(body).encode())

job_name = f"gi-promoter-batch-{stamp}"
sm.create_transform_job(
    TransformJobName=job_name,
    ModelName=model_name,
    MaxConcurrentTransforms=1,
    BatchStrategy="SingleRecord",
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{bucket}/{prefix}/input/"}},
        "ContentType": "application/json",
        "SplitType": "None",
    },
    TransformOutput={"S3OutputPath": f"s3://{bucket}/{prefix}/output/", "AssembleWith": "None"},
    TransformResources={"InstanceType": INSTANCE, "InstanceCount": 1},
)
sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=job_name)
status = sm.describe_transform_job(TransformJobName=job_name)["TransformJobStatus"]
print(job_name, status)

for name in samples:
    obj = s3.get_object(Bucket=bucket, Key=f"{prefix}/output/{name}.json.out")
    r = json.loads(obj["Body"].read())
    print(name, [round(w["probability"], 4) for w in r["window_details"]])

## Clean up

Deletes the endpoint (which stops its billing), the endpoint configuration and the model. The batch job stops billing on its own when it finishes.

In [ ]:
sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=endpoint_name)
sm.delete_model(ModelName=model_name)
print("deleted", endpoint_name)

To stop the subscription, open **Your Marketplace software** in the AWS Marketplace console, choose the product, and cancel the subscription.